# Results Synthesis Analysis

This notebook consolidates the completed thesis outputs into a compact set of thesis-ready tables, figures, and interpretation notes. It does not run transformer inference, call external APIs, or regenerate the `010x`-`060x` experiment outputs.

The canonical contextual run is the `050x` output set. Older `040x` contextual files may exist from earlier runs, but they are intentionally ignored here.


## Imports and paths

The notebook assumes it is run from the project root. All generated synthesis outputs use the `070x` prefix so they do not overwrite earlier experiment outputs.


In [1]:
from __future__ import annotations

from pathlib import Path
import math
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().resolve()
RESULTS_DIR = PROJECT_ROOT / "outputs" / "results"
TABLE_DIR = PROJECT_ROOT / "outputs" / "tables"
FIGURE_DIR = PROJECT_ROOT / "outputs" / "figures"

for directory in [RESULTS_DIR, TABLE_DIR, FIGURE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

REQUIRED_FILES = {
    "isolated_summary": RESULTS_DIR / "0302-isolated_all_models_summary.csv",
    "contextual_summary": RESULTS_DIR / "0502-contextual_all_models_summary.csv",
    "combined_summary": RESULTS_DIR / "0504-isolated_vs_contextual_summary.csv",
    "triplet_main": RESULTS_DIR / "0603-triplet_probe_main_table.csv",
    "triplet_robustness": RESULTS_DIR / "0604-triplet_probe_robustness.csv",
    "tokenization_split_pairs": TABLE_DIR / "0102-tokenization_split_pair_percentage.csv",
    "tokenization_pair_complexity": TABLE_DIR / "0106-tokenization_pair_complexity_summary.csv",
    "isolated_pair_tokenization": RESULTS_DIR / "0303-isolated_all_models_pair_similarities_with_tokenization.csv",
    "contextual_pair_tokenization": RESULTS_DIR / "0503-contextual_all_models_pair_similarities_with_tokenization.csv",
}

OUTPUT_FILES = {
    "main_results": TABLE_DIR / "0701-main_results_for_thesis.csv",
    "best_by_model_condition": TABLE_DIR / "0702-best_by_model_condition.csv",
    "contextual_gain": TABLE_DIR / "0703-contextual_gain_summary.csv",
    "pooling_layer_model": TABLE_DIR / "0704-pooling_layer_model_summary.csv",
    "tokenization_fragmentation": TABLE_DIR / "0705-tokenization_fragmentation_summary.csv",
    "triplet_probe": TABLE_DIR / "0706-triplet_probe_thesis_summary.csv",
    "main_overview_figure": FIGURE_DIR / "0701-main_results_overview.png",
    "contextual_gain_figure": FIGURE_DIR / "0702-contextual_gain_by_model.png",
    "triplet_summary_figure": FIGURE_DIR / "0703-triplet_probe_exploratory_summary.png",
    "fragmentation_overview_figure": FIGURE_DIR / "0704-tokenization_fragmentation_overview.png",
    "clean_split_spearman": TABLE_DIR / "0707-clean_vs_split_spearman_summary.csv",
    "clean_split_spearman_figure": FIGURE_DIR / "0705-clean_vs_split_spearman.png",
}

EXPECTED_MODELS = {"BERTurk", "mBERT", "XLM-R"}
EXPECTED_LAYERS = {1, 7, 12}
EXPECTED_POOLINGS = {"first", "last", "mean", "max"}
INPUT_ORDER = ["isolated", "contextual"]
MODEL_ORDER = ["BERTurk", "mBERT", "XLM-R"]
POOLING_ORDER = ["first", "last", "mean", "max"]
LAYER_ORDER = [1, 7, 12]

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
})

print(f"Project root: {PROJECT_ROOT}")


Project root: /Users/nebox/Library/CloudStorage/GoogleDrive-anebieren@gmail.com/My Drive/ORTAK/nebox/Lectures/+2nd semester/Thesis/+thesis-proJect


## Load and validate result files

The validation checks confirm that the notebook is reading the completed experiment outputs with the expected shape and configuration grid. If any of these checks fail, the synthesis should not be interpreted until the upstream output is regenerated or the expected design is intentionally revised.


In [2]:
missing_files = [path for path in REQUIRED_FILES.values() if not path.exists()]
if missing_files:
    missing_text = "\n".join(str(path) for path in missing_files)
    raise FileNotFoundError(f"Missing required synthesis inputs:\n{missing_text}")

isolated_summary = pd.read_csv(REQUIRED_FILES["isolated_summary"])
contextual_summary = pd.read_csv(REQUIRED_FILES["contextual_summary"])
combined_summary = pd.read_csv(REQUIRED_FILES["combined_summary"])
triplet_main = pd.read_csv(REQUIRED_FILES["triplet_main"])
triplet_robustness = pd.read_csv(REQUIRED_FILES["triplet_robustness"])
token_split_pairs = pd.read_csv(REQUIRED_FILES["tokenization_split_pairs"])
token_pair_complexity = pd.read_csv(REQUIRED_FILES["tokenization_pair_complexity"])
isolated_pair_tokenization = pd.read_csv(REQUIRED_FILES["isolated_pair_tokenization"])
contextual_pair_tokenization = pd.read_csv(REQUIRED_FILES["contextual_pair_tokenization"])

def require_equal(name: str, actual, expected) -> None:
    if actual != expected:
        raise AssertionError(f"{name}: expected {expected!r}, got {actual!r}")

require_equal("isolated summary row count", len(isolated_summary), 36)
require_equal("contextual summary row count", len(contextual_summary), 36)
require_equal("combined isolated/contextual row count", len(combined_summary), 72)
require_equal("triplet main table row count", len(triplet_main), 3)
require_equal("isolated pair-tokenization row count", len(isolated_pair_tokenization), 18000)
require_equal("contextual pair-tokenization row count", len(contextual_pair_tokenization), 18000)

for frame_name, frame in [
    ("isolated_summary", isolated_summary),
    ("contextual_summary", contextual_summary),
    ("combined_summary", combined_summary),
]:
    require_equal(f"{frame_name} models", set(frame["model"]), EXPECTED_MODELS)
    require_equal(f"{frame_name} layers", set(frame["layer"].astype(int)), EXPECTED_LAYERS)
    require_equal(f"{frame_name} pooling methods", set(frame["pooling"]), EXPECTED_POOLINGS)

require_equal(
    "combined input conditions",
    set(combined_summary["input_condition"]),
    {"isolated", "contextual"},
)

print("Loaded and validated canonical synthesis inputs.")
print("Ignored legacy 040x contextual files by design; 050x is the canonical contextual output set.")


Loaded and validated canonical synthesis inputs.
Ignored legacy 040x contextual files by design; 050x is the canonical contextual output set.


## Research question map

| Thesis question | Evidence used in this synthesis | Primary output |
|---|---|---|
| Which subtoken-to-word composition strategy best matches human similarity judgments? | Isolated and contextual Spearman summaries | `0701`, `0702`, `0704` |
| Does the answer change between isolated and contextual input? | Isolated-vs-contextual comparison | `0703`, `0702` |
| Which model and layer are strongest? | Model/layer/pooling grid summaries | `0702`, `0704` |
| How fragmented are the AnlamVer pairs under each tokenizer? | Tokenization split-rate and pair-complexity tables | `0705` |
| Does the morphology probe support a root-dominant interpretation? | Triplet probe headline and robustness tables | `0706` |


## Tokenization and fragmentation overview

This section summarizes fragmentation as a descriptive and moderating property of the evaluation set. It should not be interpreted as causal evidence that fragmentation alone causes lower or higher semantic alignment.


In [3]:
TOKEN_MODEL_LABELS = {
    "berturk": "BERTurk",
    "mbert": "mBERT",
    "xlmr": "XLM-R",
}

split = token_split_pairs.copy()
complexity = token_pair_complexity.copy()
split["model"] = split["model"].astype(str).str.lower().map(TOKEN_MODEL_LABELS)
complexity["model"] = complexity["model"].astype(str).str.lower().map(TOKEN_MODEL_LABELS)

tokenization_fragmentation_summary = split.merge(complexity, on="model", how="left")
tokenization_fragmentation_summary = tokenization_fragmentation_summary[
    [
        "model",
        "word_pairs",
        "clean_pairs",
        "split_pairs",
        "both_words_split_pairs",
        "split_pair_percentage",
        "both_words_split_percentage",
        "mean_pair_total_subtoken_count",
        "median_pair_total_subtoken_count",
        "mean_pair_max_subtoken_count",
        "mean_pair_avg_subtoken_char_length",
    ]
].sort_values("model", key=lambda s: s.map({m: i for i, m in enumerate(MODEL_ORDER)}))

tokenization_fragmentation_summary


,model,word_pairs,clean_pairs,split_pairs,both_words_split_pairs,split_pair_percentage,both_words_split_percentage,mean_pair_total_subtoken_count,median_pair_total_subtoken_count,mean_pair_max_subtoken_count,mean_pair_avg_subtoken_char_length
0,BERTurk,500,225,275,105,55.0,21.0,3.092,3.0,1.826,4.613029
1,mBERT,500,12,488,402,97.6,80.4,5.496,5.0,3.258,2.528752
2,XLM-R,500,93,407,219,81.4,43.8,4.030,4.0,2.438,3.658163


## Main isolated and contextual results

The main result is selected by Spearman correlation between model cosine similarities and AnlamVer human similarity ratings. The global best row is expected to duplicate the best contextual row in the current output snapshot.


In [4]:
def sort_for_display(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if "input_condition" in out:
        out["input_condition"] = pd.Categorical(out["input_condition"], INPUT_ORDER, ordered=True)
    if "model" in out:
        out["model"] = pd.Categorical(out["model"], MODEL_ORDER, ordered=True)
    if "pooling" in out:
        out["pooling"] = pd.Categorical(out["pooling"], POOLING_ORDER, ordered=True)
    return out

combined = combined_summary.copy()
combined["layer"] = combined["layer"].astype(int)

def best_configuration(df: pd.DataFrame, label: str) -> pd.DataFrame:
    row = df.sort_values("spearman_rho", ascending=False).head(1).copy()
    row.insert(0, "result_label", label)
    return row

main_results_for_thesis = pd.concat(
    [
        best_configuration(combined.query("input_condition == 'isolated'"), "best_isolated"),
        best_configuration(combined.query("input_condition == 'contextual'"), "best_contextual"),
        best_configuration(combined, "global_best"),
    ],
    ignore_index=True,
)
main_results_for_thesis = main_results_for_thesis[
    [
        "result_label",
        "input_condition",
        "model",
        "model_name",
        "layer",
        "pooling",
        "spearman_rho",
        "p_value",
        "n_pairs",
    ]
]

main_results_for_thesis


,result_label,input_condition,model,model_name,layer,pooling,spearman_rho,p_value,n_pairs
0,best_isolated,isolated,BERTurk,dbmdz/bert-base-turkish-cased,1,first,0.426694,1.533853e-23,500
1,best_contextual,contextual,BERTurk,dbmdz/bert-base-turkish-cased,7,first,0.527538,3.630230e-37,500
2,global_best,contextual,BERTurk,dbmdz/bert-base-turkish-cased,7,first,0.527538,3.630230e-37,500


In [5]:
best_by_model_condition = (
    combined.sort_values("spearman_rho", ascending=False)
    .groupby(["input_condition", "model"], as_index=False, sort=False)
    .head(1)
    .copy()
)
best_by_model_condition = sort_for_display(best_by_model_condition).sort_values(
    ["input_condition", "model"]
)
best_by_model_condition = best_by_model_condition[
    [
        "input_condition",
        "model",
        "model_name",
        "layer",
        "pooling",
        "spearman_rho",
        "p_value",
        "n_pairs",
    ]
].reset_index(drop=True)

best_by_model_condition


,input_condition,model,model_name,layer,pooling,spearman_rho,p_value,n_pairs
0,isolated,BERTurk,dbmdz/bert-base-turkish-cased,1,first,0.426694,1.533853e-23,500
1,isolated,mBERT,bert-base-multilingual-cased,12,first,0.154553,5.241110e-04,500
2,isolated,XLM-R,xlm-roberta-base,1,first,0.121172,6.673702e-03,500
3,contextual,BERTurk,dbmdz/bert-base-turkish-cased,7,first,0.527538,3.630230e-37,500
4,contextual,mBERT,bert-base-multilingual-cased,7,last,0.244138,3.215030e-08,500
5,contextual,XLM-R,xlm-roberta-base,7,first,0.247803,1.965753e-08,500


## Pooling, layer, and model synthesis

The table below averages Spearman correlations across complementary views of the configuration grid. This is useful for thesis writing because it separates the overall model, layer, and pooling patterns from the single best configuration.


In [6]:
def aggregate_grid(df: pd.DataFrame, group_cols: list[str], aggregation_level: str) -> pd.DataFrame:
    out = (
        df.groupby(group_cols, dropna=False)
        .agg(
            mean_spearman_rho=("spearman_rho", "mean"),
            sd_spearman_rho=("spearman_rho", "std"),
            min_spearman_rho=("spearman_rho", "min"),
            max_spearman_rho=("spearman_rho", "max"),
            n_configurations=("spearman_rho", "size"),
        )
        .reset_index()
    )
    out.insert(0, "aggregation_level", aggregation_level)
    for column in ["input_condition", "model", "layer", "pooling"]:
        if column not in out.columns:
            out[column] = "all"
    return out[
        [
            "aggregation_level",
            "input_condition",
            "model",
            "layer",
            "pooling",
            "mean_spearman_rho",
            "sd_spearman_rho",
            "min_spearman_rho",
            "max_spearman_rho",
            "n_configurations",
        ]
    ]

pooling_layer_model_summary = pd.concat(
    [
        aggregate_grid(combined, ["input_condition", "model"], "model_overall"),
        aggregate_grid(combined, ["input_condition", "layer"], "layer_overall"),
        aggregate_grid(combined, ["input_condition", "pooling"], "pooling_overall"),
        aggregate_grid(combined, ["input_condition", "model", "layer"], "model_layer"),
        aggregate_grid(combined, ["input_condition", "model", "pooling"], "model_pooling"),
    ],
    ignore_index=True,
)
pooling_layer_model_summary = pooling_layer_model_summary.sort_values(
    ["aggregation_level", "input_condition", "model", "layer", "pooling"]
).reset_index(drop=True)

pooling_layer_model_summary.head(15)


,aggregation_level,input_condition,model,layer,pooling,mean_spearman_rho,sd_spearman_rho,min_spearman_rho,max_spearman_rho,n_configurations
0,layer_overall,contextual,all,1,all,0.105693,0.183863,-0.136425,0.455468,12
1,layer_overall,contextual,all,7,all,0.250113,0.151617,0.050445,0.527538,12
2,layer_overall,contextual,all,12,all,0.222478,0.154899,-0.031379,0.452649,12
3,layer_overall,isolated,all,1,all,0.122348,0.175107,-0.119305,0.426694,12
4,layer_overall,isolated,all,7,all,0.122393,0.134831,-0.052757,0.315051,12
5,layer_overall,isolated,all,12,all,0.173183,0.133747,-0.005553,0.356264,12
6,model_layer,contextual,BERTurk,1,all,0.316843,0.111788,0.183439,0.455468,4
7,model_layer,contextual,BERTurk,7,all,0.426280,0.078018,0.337657,0.527538,4
8,model_layer,contextual,BERTurk,12,all,0.415884,0.046907,0.352732,0.452649,4
9,model_layer,contextual,XLM-R,1,all,0.022415,0.107996,-0.088612,0.136591,4


## Contextual gain analysis

This comparison keeps model, layer, and pooling fixed, then subtracts the isolated Spearman correlation from the contextual Spearman correlation. Positive values indicate that sentence-context averaging improved alignment with human similarity ratings for that exact configuration.


In [7]:
iso = combined.query("input_condition == 'isolated'").copy()
ctx = combined.query("input_condition == 'contextual'").copy()

contextual_gain_summary = iso.merge(
    ctx,
    on=["model", "model_name", "layer", "pooling"],
    suffixes=("_isolated", "_contextual"),
    validate="one_to_one",
)
contextual_gain_summary = contextual_gain_summary.rename(
    columns={
        "spearman_rho_isolated": "isolated_spearman_rho",
        "spearman_rho_contextual": "contextual_spearman_rho",
        "p_value_isolated": "isolated_p_value",
        "p_value_contextual": "contextual_p_value",
        "n_pairs_isolated": "isolated_n_pairs",
        "n_pairs_contextual": "contextual_n_pairs",
    }
)
contextual_gain_summary["contextual_minus_isolated"] = (
    contextual_gain_summary["contextual_spearman_rho"]
    - contextual_gain_summary["isolated_spearman_rho"]
)
contextual_gain_summary["isolated_abs_denominator"] = contextual_gain_summary[
    "isolated_spearman_rho"
].abs().replace(0, np.nan)
contextual_gain_summary["relative_contextual_gain"] = (
    contextual_gain_summary["contextual_minus_isolated"]
    / contextual_gain_summary["isolated_abs_denominator"]
)
contextual_gain_summary = contextual_gain_summary[
    [
        "model",
        "model_name",
        "layer",
        "pooling",
        "isolated_spearman_rho",
        "contextual_spearman_rho",
        "contextual_minus_isolated",
        "relative_contextual_gain",
        "isolated_p_value",
        "contextual_p_value",
        "isolated_n_pairs",
        "contextual_n_pairs",
    ]
].sort_values("contextual_minus_isolated", ascending=False)

contextual_gain_summary.head(12)


,model,model_name,layer,pooling,isolated_spearman_rho,contextual_spearman_rho,contextual_minus_isolated,relative_contextual_gain,isolated_p_value,contextual_p_value,isolated_n_pairs,contextual_n_pairs
4,BERTurk,dbmdz/bert-base-turkish-cased,7,first,0.315051,0.527538,0.212488,0.674456,5.531052e-13,3.630230e-37,500,500
28,XLM-R,xlm-roberta-base,7,first,0.059575,0.247803,0.188229,3.159549,1.835268e-01,1.965753e-08,500,500
17,mBERT,bert-base-multilingual-cased,7,last,0.073259,0.244138,0.170878,2.332509,1.017916e-01,3.215030e-08,500,500
18,mBERT,bert-base-multilingual-cased,7,mean,0.034143,0.182747,0.148604,4.352440,4.462003e-01,3.943257e-05,500,500
16,mBERT,bert-base-multilingual-cased,7,first,0.090440,0.232516,0.142076,1.570953,4.324111e-02,1.452816e-07,500,500
5,BERTurk,dbmdz/bert-base-turkish-cased,7,last,0.287228,0.413901,0.126673,0.441019,5.951604e-11,4.085490e-22,500,500
29,XLM-R,xlm-roberta-base,7,last,0.067586,0.189823,0.122236,1.808597,1.312426e-01,1.929885e-05,500,500
6,BERTurk,dbmdz/bert-base-turkish-cased,7,mean,0.305104,0.426025,0.120921,0.396327,3.120747e-12,1.827596e-23,500,500
30,XLM-R,xlm-roberta-base,7,mean,-0.052757,0.061881,0.114639,2.172944,2.389737e-01,1.671041e-01,500,500
34,XLM-R,xlm-roberta-base,12,mean,0.014073,0.114104,0.100031,7.107966,7.535889e-01,1.066783e-02,500,500


## Fragmentation synthesis

The synthesis table below should be read alongside the detailed fragmentation plots generated in the isolated and contextual experiment notebooks. In the thesis text, fragmentation should be framed as descriptive evidence about tokenizer behavior and as a possible moderator of pooling reliability.


In [8]:
tokenization_fragmentation_summary


,model,word_pairs,clean_pairs,split_pairs,both_words_split_pairs,split_pair_percentage,both_words_split_percentage,mean_pair_total_subtoken_count,median_pair_total_subtoken_count,mean_pair_max_subtoken_count,mean_pair_avg_subtoken_char_length
0,BERTurk,500,225,275,105,55.0,21.0,3.092,3.0,1.826,4.613029
1,mBERT,500,12,488,402,97.6,80.4,5.496,5.0,3.258,2.528752
2,XLM-R,500,93,407,219,81.4,43.8,4.030,4.0,2.438,3.658163


## Triplet probe synthesis

The triplet probe is included as exploratory mechanistic evidence. Positive `mean_delta` values indicate that embeddings were closer for words sharing a root than for words sharing only a suffix. This supports a root-dominant interpretation of the strong first-pooling result, but it does not replace the AnlamVer Spearman-correlation analysis.


In [9]:
def bool_series(series: pd.Series) -> pd.Series:
    if series.dtype == bool:
        return series
    return series.astype(str).str.lower().isin({"true", "1", "yes"})

headline = triplet_main.copy()
headline.insert(0, "summary_source", "headline_all_triplets")
headline.insert(1, "robustness_subset", "all_triplets")
headline.insert(2, "subset_description", "Headline all-triplet summary from 0603")

robustness_best = triplet_robustness[bool_series(triplet_robustness["is_best_in_subset"])].copy()
robustness_best = robustness_best.query("robustness_subset != 'all_triplets'").copy()
robustness_best.insert(0, "summary_source", "robustness_best_by_subset_model")

triplet_probe_thesis_summary = pd.concat(
    [headline, robustness_best],
    ignore_index=True,
    sort=False,
)
triplet_probe_thesis_summary = triplet_probe_thesis_summary[
    [
        "summary_source",
        "robustness_subset",
        "subset_description",
        "model",
        "model_name",
        "layer",
        "pooling",
        "n",
        "mean_sim_same_root",
        "mean_sim_same_suffix",
        "mean_delta",
        "ci95_low",
        "ci95_high",
        "wilcoxon_p",
    ]
].sort_values(["summary_source", "robustness_subset", "model"]).reset_index(drop=True)

triplet_probe_thesis_summary


,summary_source,robustness_subset,subset_description,model,model_name,layer,pooling,n,mean_sim_same_root,mean_sim_same_suffix,mean_delta,ci95_low,ci95_high,wilcoxon_p
0,headline_all_triplets,all_triplets,Headline all-triplet summary from 0603,BERTurk,dbmdz/bert-base-turkish-cased,1,first,60,0.751026,0.381081,0.369945,0.324008,0.414490,1.629556e-11
1,headline_all_triplets,all_triplets,Headline all-triplet summary from 0603,XLM-R,xlm-roberta-base,1,first,60,0.955303,0.568363,0.386940,0.354319,0.413862,1.994567e-11
2,headline_all_triplets,all_triplets,Headline all-triplet summary from 0603,mBERT,bert-base-multilingual-cased,1,first,60,0.979330,0.342685,0.636645,0.613137,0.658944,1.629556e-11
3,robustness_best_by_subset_model,berturk_all_split_triplets,"Only triplets where BERTurk splits A_x, A_y, a...",BERTurk,dbmdz/bert-base-turkish-cased,1,first,13,0.972161,0.341644,0.630517,0.572754,0.673118,2.441406e-04
4,robustness_best_by_subset_model,berturk_all_split_triplets,"Only triplets where BERTurk splits A_x, A_y, a...",XLM-R,xlm-roberta-base,1,first,13,0.986329,0.604771,0.381558,0.359105,0.402504,2.441406e-04
5,robustness_best_by_subset_model,berturk_all_split_triplets,"Only triplets where BERTurk splits A_x, A_y, a...",mBERT,bert-base-multilingual-cased,1,first,13,0.961407,0.414225,0.547181,0.503756,0.590585,2.441406e-04
6,robustness_best_by_subset_model,berturk_shared_final_subtoken,Only triplets where BERTurk A_x and B_x share ...,BERTurk,dbmdz/bert-base-turkish-cased,1,first,18,0.879419,0.396472,0.482947,0.418446,0.544835,7.629395e-06
7,robustness_best_by_subset_model,berturk_shared_final_subtoken,Only triplets where BERTurk A_x and B_x share ...,XLM-R,xlm-roberta-base,1,first,18,0.987830,0.555633,0.432197,0.414496,0.449590,7.629395e-06
8,robustness_best_by_subset_model,berturk_shared_final_subtoken,Only triplets where BERTurk A_x and B_x share ...,mBERT,bert-base-multilingual-cased,1,first,18,0.974943,0.386566,0.588377,0.547133,0.627934,7.629395e-06
9,robustness_best_by_subset_model,model_all_three_split,"Only rows where the current model splits A_x, ...",BERTurk,dbmdz/bert-base-turkish-cased,1,first,13,0.972161,0.341644,0.630517,0.572754,0.673118,2.441406e-04


## Thesis-ready tables

The following cell writes the `070x` synthesis tables. These files are derived from existing result CSVs and do not modify upstream experiment outputs.


In [10]:
table_outputs = {
    OUTPUT_FILES["main_results"]: main_results_for_thesis,
    OUTPUT_FILES["best_by_model_condition"]: best_by_model_condition,
    OUTPUT_FILES["contextual_gain"]: contextual_gain_summary,
    OUTPUT_FILES["pooling_layer_model"]: pooling_layer_model_summary,
    OUTPUT_FILES["tokenization_fragmentation"]: tokenization_fragmentation_summary,
    OUTPUT_FILES["triplet_probe"]: triplet_probe_thesis_summary,
}

for path, frame in table_outputs.items():
    frame.to_csv(path, index=False, encoding="utf-8")
    print(f"Saved {len(frame):>3} rows -> {path.relative_to(PROJECT_ROOT)}")


Saved   3 rows -> outputs/tables/0701-main_results_for_thesis.csv
Saved   6 rows -> outputs/tables/0702-best_by_model_condition.csv
Saved  36 rows -> outputs/tables/0703-contextual_gain_summary.csv
Saved  62 rows -> outputs/tables/0704-pooling_layer_model_summary.csv
Saved   3 rows -> outputs/tables/0705-tokenization_fragmentation_summary.csv
Saved  12 rows -> outputs/tables/0706-triplet_probe_thesis_summary.csv


## Thesis-ready figures

These figures summarize the main thesis patterns. The earlier experiment notebooks remain the source for more detailed heatmaps and fragmentation diagnostics.


In [11]:
palette_input = {"isolated": "#4C78A8", "contextual": "#F58518"}
palette_pooling = {
    "first": "#4C78A8",
    "last": "#54A24B",
    "mean": "#F58518",
    "max": "#B279A2",
}
palette_triplet = {"BERTurk": "#4C78A8", "mBERT": "#54A24B", "XLM-R": "#F58518"}

fig, ax = plt.subplots(figsize=(7.2, 4.2))
plot_data = best_by_model_condition.copy()
plot_data["model"] = pd.Categorical(plot_data["model"], MODEL_ORDER, ordered=True)
plot_data["input_condition"] = pd.Categorical(plot_data["input_condition"], INPUT_ORDER, ordered=True)
sns.barplot(
    data=plot_data,
    x="model",
    y="spearman_rho",
    hue="input_condition",
    order=MODEL_ORDER,
    hue_order=INPUT_ORDER,
    palette=palette_input,
    ax=ax,
)
ax.set_title("Best Spearman correlation by model and input condition")
ax.set_xlabel("Model")
ax.set_ylabel("Best Spearman rho")
ax.set_ylim(0, max(0.58, plot_data["spearman_rho"].max() + 0.05))
ax.legend(title="Input condition", frameon=True)
for container in ax.containers:
    ax.bar_label(container, fmt="%.3f", padding=2, fontsize=8)
fig.tight_layout()
fig.savefig(OUTPUT_FILES["main_overview_figure"], bbox_inches="tight")
plt.close(fig)

layer7_gain = contextual_gain_summary.query("layer == 7").copy()
layer7_gain["model"] = pd.Categorical(layer7_gain["model"], MODEL_ORDER, ordered=True)
layer7_gain["pooling"] = pd.Categorical(layer7_gain["pooling"], POOLING_ORDER, ordered=True)
fig, ax = plt.subplots(figsize=(8.0, 4.3))
sns.barplot(
    data=layer7_gain,
    x="model",
    y="contextual_minus_isolated",
    hue="pooling",
    order=MODEL_ORDER,
    hue_order=POOLING_ORDER,
    palette=palette_pooling,
    ax=ax,
)
ax.axhline(0, color="black", linewidth=0.9)
ax.set_title("Contextual gain at layer 7")
ax.set_xlabel("Model")
ax.set_ylabel("Contextual minus isolated Spearman rho")
ax.legend(title="Pooling", frameon=True, ncols=4, loc="upper center", bbox_to_anchor=(0.5, -0.15))
fig.tight_layout()
fig.savefig(OUTPUT_FILES["contextual_gain_figure"], bbox_inches="tight")
plt.close(fig)

triplet_plot = triplet_main.copy()
triplet_plot["model"] = pd.Categorical(triplet_plot["model"], MODEL_ORDER, ordered=True)
triplet_plot = triplet_plot.sort_values("model")
yerr = np.vstack([
    triplet_plot["mean_delta"] - triplet_plot["ci95_low"],
    triplet_plot["ci95_high"] - triplet_plot["mean_delta"],
])
fig, ax = plt.subplots(figsize=(6.8, 4.2))
colors = [palette_triplet[str(model)] for model in triplet_plot["model"]]
ax.bar(triplet_plot["model"].astype(str), triplet_plot["mean_delta"], color=colors, yerr=yerr, capsize=4)
ax.axhline(0, color="black", linewidth=0.9)
ax.set_title("Triplet probe: root advantage over suffix sharing")
ax.set_xlabel("Model")
ax.set_ylabel("Mean delta: same-root minus same-suffix cosine")
for index, row in enumerate(triplet_plot.itertuples()):
    ax.text(index, row.mean_delta + 0.025, f"{row.mean_delta:.3f}", ha="center", va="bottom", fontsize=8)
fig.tight_layout()
fig.savefig(OUTPUT_FILES["triplet_summary_figure"], bbox_inches="tight")
plt.close(fig)

for key in ["main_overview_figure", "contextual_gain_figure", "triplet_summary_figure"]:
    print(f"Saved figure -> {OUTPUT_FILES[key].relative_to(PROJECT_ROOT)}")


Saved figure -> outputs/figures/0701-main_results_overview.png
Saved figure -> outputs/figures/0702-contextual_gain_by_model.png
Saved figure -> outputs/figures/0703-triplet_probe_exploratory_summary.png


## Results interpretation notes

The following notes are generated from the loaded result tables. They are phrased for thesis drafting, but they should still be edited into the final narrative rather than copied mechanically.


In [12]:
best_isolated = main_results_for_thesis.query("result_label == 'best_isolated'").iloc[0]
best_contextual = main_results_for_thesis.query("result_label == 'best_contextual'").iloc[0]
best_gain = contextual_gain_summary.sort_values("contextual_minus_isolated", ascending=False).iloc[0]

notes = [
    (
        "Main answer: the strongest configuration is "
        f"{best_contextual.input_condition} {best_contextual.model}, layer {int(best_contextual.layer)}, "
        f"{best_contextual.pooling} pooling (Spearman rho = {best_contextual.spearman_rho:.4f})."
    ),
    (
        "Isolated condition: the strongest isolated configuration is "
        f"{best_isolated.model}, layer {int(best_isolated.layer)}, {best_isolated.pooling} pooling "
        f"(Spearman rho = {best_isolated.spearman_rho:.4f})."
    ),
    (
        "Pooling interpretation: first pooling performs strongly across the main headline results, "
        "contrary to the initial expectation that mean pooling would generally dominate."
    ),
    (
        "Layer and context interpretation: the largest contextual improvement occurs for "
        f"{best_gain.model}, layer {int(best_gain.layer)}, {best_gain.pooling} pooling "
        f"(delta = {best_gain.contextual_minus_isolated:.4f}), supporting the value of sentence-context averaging, especially around the middle layer."
    ),
    (
        "Fragmentation interpretation: tokenization fragmentation differs strongly by tokenizer and should be framed as a descriptive/moderating property, not as direct causal evidence."
    ),
    (
        "Triplet interpretation: all headline triplet mean deltas are positive, supporting a root-dominant interpretation of first pooling; because stricter tokenization subsets are smaller, this result should remain exploratory."
    ),
]

for note in notes:
    print(f"- {note}")


- Main answer: the strongest configuration is contextual BERTurk, layer 7, first pooling (Spearman rho = 0.5275).
- Isolated condition: the strongest isolated configuration is BERTurk, layer 1, first pooling (Spearman rho = 0.4267).
- Pooling interpretation: first pooling performs strongly across the main headline results, contrary to the initial expectation that mean pooling would generally dominate.
- Layer and context interpretation: the largest contextual improvement occurs for BERTurk, layer 7, first pooling (delta = 0.2125), supporting the value of sentence-context averaging, especially around the middle layer.
- Fragmentation interpretation: tokenization fragmentation differs strongly by tokenizer and should be framed as a descriptive/moderating property, not as direct causal evidence.
- Triplet interpretation: all headline triplet mean deltas are positive, supporting a root-dominant interpretation of first pooling; because stricter tokenization subsets are smaller, this result 

## Additional fragmentation figures

These final figures are intended for the thesis tokenization and fragmentation discussion. They focus on subtoken count and clean/split status because these directly affect the pooling problem. Subtoken length remains a descriptive tokenizer diagnostic rather than a main explanatory figure.

In [ ]:
# Figure 0704: tokenizer fragmentation overview based on subtoken count.
fragmentation_plot = tokenization_fragmentation_summary.copy()
fragmentation_plot["model"] = pd.Categorical(fragmentation_plot["model"], MODEL_ORDER, ordered=True)
fragmentation_plot = fragmentation_plot.sort_values("model")

fig, axes = plt.subplots(1, 3, figsize=(12.0, 3.8))
fragmentation_palette = {"BERTurk": "#4C78A8", "mBERT": "#54A24B", "XLM-R": "#F58518"}

panels = [
    ("split_pair_percentage", "Split pairs (%)", "Pairs where at least one word is split"),
    ("both_words_split_percentage", "Both words split (%)", "Pairs where both words are split"),
    ("mean_pair_total_subtoken_count", "Mean total subtokens", "Subtokens per word pair"),
]
for ax, (column, ylabel, title) in zip(axes, panels):
    sns.barplot(
        data=fragmentation_plot,
        x="model",
        y=column,
        hue="model",
        order=MODEL_ORDER,
        hue_order=MODEL_ORDER,
        palette=fragmentation_palette,
        legend=False,
        ax=ax,
    )
    ax.set_title(title)
    ax.set_xlabel("Model tokenizer")
    ax.set_ylabel(ylabel)
    if "percentage" in column:
        ax.set_ylim(0, 105)
        label_fmt = "%.1f"
    else:
        ax.set_ylim(0, max(6.0, fragmentation_plot[column].max() + 0.6))
        label_fmt = "%.2f"
    for container in ax.containers:
        ax.bar_label(container, fmt=label_fmt, padding=2, fontsize=8)

fig.suptitle("Tokenization fragmentation across AnlamVer word pairs", y=1.03, fontsize=13)
fig.tight_layout()
fig.savefig(OUTPUT_FILES["fragmentation_overview_figure"], bbox_inches="tight")
plt.close(fig)


def _spearman_for_subset(frame: pd.DataFrame, condition: str, best_row: pd.Series, split_column: str, split_value: bool) -> dict:
    subset = frame[
        (frame["model"] == best_row["model"])
        & (frame["layer"].astype(int) == int(best_row["layer"]))
        & (frame["pooling"] == best_row["pooling"])
        & (frame[split_column] == split_value)
    ].copy()
    return {
        "input_condition": condition,
        "model": best_row["model"],
        "layer": int(best_row["layer"]),
        "pooling": best_row["pooling"],
        "fragmentation_group": "clean" if split_column == "clean_pair" else "split",
        "n_pairs": int(len(subset)),
        "spearman_rho": float(subset["Sim"].corr(subset["cosine_similarity"], method="spearman")),
        "mean_pair_total_subtoken_count": float(subset["pair_total_subtoken_count"].mean()),
        "mean_pair_max_subtoken_count": float(subset["pair_max_subtoken_count"].mean()),
    }

best_isolated_for_fragmentation = main_results_for_thesis.query("result_label == 'best_isolated'").iloc[0]
best_contextual_for_fragmentation = main_results_for_thesis.query("result_label == 'best_contextual'").iloc[0]

clean_split_rows = []
for frame, condition, best_row in [
    (isolated_pair_tokenization, "isolated", best_isolated_for_fragmentation),
    (contextual_pair_tokenization, "contextual", best_contextual_for_fragmentation),
]:
    clean_split_rows.append(_spearman_for_subset(frame, condition, best_row, "clean_pair", True))
    clean_split_rows.append(_spearman_for_subset(frame, condition, best_row, "split_pair", True))

clean_split_spearman_summary = pd.DataFrame(clean_split_rows)
clean_split_spearman_summary["input_condition"] = pd.Categorical(
    clean_split_spearman_summary["input_condition"], INPUT_ORDER, ordered=True
)
clean_split_spearman_summary["fragmentation_group"] = pd.Categorical(
    clean_split_spearman_summary["fragmentation_group"], ["clean", "split"], ordered=True
)
clean_split_spearman_summary = clean_split_spearman_summary.sort_values(
    ["input_condition", "fragmentation_group"]
)
clean_split_spearman_summary.to_csv(OUTPUT_FILES["clean_split_spearman"], index=False)

# Figure 0705: clean vs split semantic alignment for the strongest isolated/contextual configurations.
fig, ax = plt.subplots(figsize=(6.8, 4.1))
sns.barplot(
    data=clean_split_spearman_summary,
    x="input_condition",
    y="spearman_rho",
    hue="fragmentation_group",
    order=INPUT_ORDER,
    hue_order=["clean", "split"],
    palette={"clean": "#4C78A8", "split": "#E45756"},
    ax=ax,
)
ax.set_title("Clean vs split pairs for the strongest BERTurk configurations")
ax.set_xlabel("Input condition")
ax.set_ylabel("Spearman rho")
ax.set_ylim(0, max(0.65, clean_split_spearman_summary["spearman_rho"].max() + 0.06))
ax.legend(title="Pair type", frameon=True)
for container in ax.containers:
    ax.bar_label(container, fmt="%.3f", padding=2, fontsize=8)
fig.tight_layout()
fig.savefig(OUTPUT_FILES["clean_split_spearman_figure"], bbox_inches="tight")
plt.close(fig)

print(f"Saved figure -> {OUTPUT_FILES['fragmentation_overview_figure'].relative_to(PROJECT_ROOT)}")
print(f"Saved table  -> {OUTPUT_FILES['clean_split_spearman'].relative_to(PROJECT_ROOT)}")
print(f"Saved figure -> {OUTPUT_FILES['clean_split_spearman_figure'].relative_to(PROJECT_ROOT)}")
clean_split_spearman_summary


## Output verification

This final cell checks that every planned synthesis table and figure exists, and that the current headline values match the expected output snapshot.


In [13]:
for label, path in OUTPUT_FILES.items():
    status = "OK" if path.exists() else "MISSING"
    print(f"{status:7} {label:32} {path.relative_to(PROJECT_ROOT)}")

missing_outputs = [path for path in OUTPUT_FILES.values() if not path.exists()]
if missing_outputs:
    raise FileNotFoundError("Missing generated outputs: " + ", ".join(str(path) for path in missing_outputs))

def assert_approx(label: str, actual: float, expected: float, tolerance: float = 5e-4) -> None:
    if not math.isclose(actual, expected, abs_tol=tolerance):
        raise AssertionError(f"{label}: expected about {expected}, got {actual}")

if not (best_isolated.model == "BERTurk" and int(best_isolated.layer) == 1 and best_isolated.pooling == "first"):
    raise AssertionError("Best isolated configuration does not match the expected current snapshot.")
if not (best_contextual.model == "BERTurk" and int(best_contextual.layer) == 7 and best_contextual.pooling == "first"):
    raise AssertionError("Best contextual configuration does not match the expected current snapshot.")

assert_approx("best isolated rho", float(best_isolated.spearman_rho), 0.4267)
assert_approx("best contextual rho", float(best_contextual.spearman_rho), 0.5275)

if not (triplet_main["mean_delta"] > 0).all():
    raise AssertionError("Expected positive headline triplet deltas for all models.")

print("All synthesis outputs and headline checks passed.")


OK      main_results                     outputs/tables/0701-main_results_for_thesis.csv
OK      best_by_model_condition          outputs/tables/0702-best_by_model_condition.csv
OK      contextual_gain                  outputs/tables/0703-contextual_gain_summary.csv
OK      pooling_layer_model              outputs/tables/0704-pooling_layer_model_summary.csv
OK      tokenization_fragmentation       outputs/tables/0705-tokenization_fragmentation_summary.csv
OK      triplet_probe                    outputs/tables/0706-triplet_probe_thesis_summary.csv
OK      main_overview_figure             outputs/figures/0701-main_results_overview.png
OK      contextual_gain_figure           outputs/figures/0702-contextual_gain_by_model.png
OK      triplet_summary_figure           outputs/figures/0703-triplet_probe_exploratory_summary.png
All synthesis outputs and headline checks passed.
